## ***Genetic Disorder Subclass Screening System***

**INSTALL LIBRARIES**

In [ ]:
!pip -q install streamlit scikit-learn pandas numpy joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 75.0 MB/s eta 0:00:00


**UPLOAD CSV DATASET**

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Upload the dataset CSV file.")
DATA_PATH = csv_files[0]
print("Using:", DATA_PATH)


Saving train_genetic_disorders.csv to train_genetic_disorders.csv
Using: train_genetic_disorders.csv


**LOAD AND INSPECT**

In [ ]:
import pandas as pd, numpy as np, os, joblib
data = pd.read_csv(DATA_PATH)
print("Original shape:", data.shape)
print(data.columns.tolist())
display(data.head())


Original shape: (22083, 45)
['Patient Id', 'Patient Age', "Genes in mother's side", 'Inherited from father', 'Maternal gene', 'Paternal gene', 'Blood cell count (mcL)', 'Patient First Name', 'Family Name', "Father's name", "Mother's age", "Father's age", 'Institute Name', 'Location of Institute', 'Status', 'Respiratory Rate (breaths/min)', 'Heart Rate (rates/min', 'Test 1', 'Test 2', 'Test 3', 'Test 4', 'Test 5', 'Parental consent', 'Follow-up', 'Gender', 'Birth asphyxia', 'Autopsy shows birth defect (if applicable)', 'Place of birth', 'Folic acid details (peri-conceptional)', 'H/O serious maternal illness', 'H/O radiation exposure (x-ray)', 'H/O substance abuse', 'Assisted conception IVF/ART', 'History of anomalies in previous pregnancies', 'No. of previous abortion', 'Birth defects', 'White Blood cell count (thousand per microliter)', 'Blood test result', 'Symptom 1', 'Symptom 2', 'Symptom 3', 'Symptom 4', 'Symptom 5', 'Genetic Disorder', 'Disorder Subclass']


,Patient Id,Patient Age,Genes in mother's side,Inherited from father,Maternal gene,Paternal gene,Blood cell count (mcL),Patient First Name,Family Name,Father's name,...,Birth defects,White Blood cell count (thousand per microliter),Blood test result,Symptom 1,Symptom 2,Symptom 3,Symptom 4,Symptom 5,Genetic Disorder,Disorder Subclass
0,PID0x6418,2.0,Yes,No,Yes,No,4.760603,Richard,NaN,Larre,...,NaN,9.857562,NaN,1.0,1.0,1.0,1.0,1.0,Mitochondrial genetic inheritance disorders,Leber's hereditary optic neuropathy
1,PID0x25d5,4.0,Yes,Yes,No,No,4.910669,Mike,NaN,Brycen,...,Multiple,5.522560,normal,1.0,NaN,1.0,1.0,0.0,NaN,Cystic fibrosis
2,PID0x4a82,6.0,Yes,No,No,No,4.893297,Kimberly,NaN,Nashon,...,Singular,NaN,normal,0.0,1.0,1.0,1.0,1.0,Multifactorial genetic inheritance disorders,Diabetes
3,PID0x4ac8,12.0,Yes,No,Yes,No,4.705280,Jeffery,Hoelscher,Aayaan,...,Singular,7.919321,inconclusive,0.0,0.0,1.0,0.0,0.0,Mitochondrial genetic inheritance disorders,Leigh syndrome
4,PID0x1bf7,11.0,Yes,No,NaN,Yes,4.720703,Johanna,Stutzman,Suave,...,Multiple,4.098210,NaN,0.0,0.0,0.0,0.0,NaN,Multifactorial genetic inheritance disorders,Cancer


**CLEAN AND SET TARGET**

In [ ]:
TARGET = "Disorder Subclass"
if TARGET not in data.columns:
    raise ValueError(f"'{TARGET}' not found. Columns: {data.columns.tolist()}")

df = data.copy().replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET]).drop_duplicates()

remove_cols = [
    c for c in df.columns
    if any(word in c.lower() for word in ["patient id", "patient name", "institution", "hospital id", "record id"])
]
if "Genetic Disorder" in df.columns:
    remove_cols.append("Genetic Disorder")
remove_cols = list(dict.fromkeys(c for c in remove_cols if c != TARGET))
df = df.drop(columns=remove_cols, errors="ignore").dropna(axis=1, how="all")

print("Clean shape:", df.shape)
print("Removed columns:", remove_cols)
display(df[TARGET].value_counts().to_frame("Count"))


Clean shape: (18943, 43)
Removed columns: ['Patient Id', 'Genetic Disorder']


,Count
Disorder Subclass,
Leigh syndrome,4893
Mitochondrial myopathy,4202
Cystic fibrosis,3277
Tay-Sachs,2690
Diabetes,1743
Hemochromatosis,1291
Leber's hereditary optic neuropathy,607
Alzheimer's,148
Cancer,92


**TRAIN AND COMPARE SVM KERNELS**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(str)

# Convert columns that are mostly numeric.
for col in X.columns:
    converted = pd.to_numeric(X[col], errors="coerce")
    if converted.notna().mean() >= 0.80:
        X[col] = converted

empty_cols = [c for c in X.columns if X[c].isna().all()]
X = X.drop(columns=empty_cols)
print("Dropped all-empty columns:", empty_cols)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Determine a safe feature-selection k using training data only.
temp = preprocessor.fit_transform(X_train, y_train)
k = min(20, temp.shape[1])

results = []
models = {}
for kernel in ["linear", "poly", "rbf"]:
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("svm", SVC(kernel=kernel, class_weight="balanced"))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "Kernel": kernel,
        "Accuracy": accuracy_score(y_test, pred),
        "F1-Score": f1_score(y_test, pred, average="weighted", zero_division=0)
    })
    models[kernel] = pipe

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
best_kernel = results_df.loc[0, "Kernel"]
model = models[best_kernel]
print("Best kernel:", best_kernel)
display(results_df.style.format({"Accuracy":"{:.2%}", "F1-Score":"{:.2%}"}))
print(classification_report(y_test, model.predict(X_test), zero_division=0))


Dropped all-empty columns: []


/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [0 0 0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [0 0 0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [0 0 0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, U

Best kernel: linear


,Kernel,Accuracy,F1-Score
0,linear,33.97%,34.45%
1,rbf,33.41%,33.79%
2,poly,31.35%,32.03%


                                     precision    recall  f1-score   support

                        Alzheimer's       0.12      0.80      0.20        30
                             Cancer       0.07      0.61      0.12        18
                    Cystic fibrosis       0.41      0.34      0.37       655
                           Diabetes       0.32      0.27      0.29       349
                    Hemochromatosis       0.26      0.42      0.32       258
Leber's hereditary optic neuropathy       0.21      0.31      0.25       121
                     Leigh syndrome       0.45      0.46      0.45       979
             Mitochondrial myopathy       0.39      0.22      0.28       841
                          Tay-Sachs       0.31      0.29      0.30       538

                           accuracy                           0.34      3789
                          macro avg       0.28      0.41      0.29      3789
                       weighted avg       0.37      0.34      0.34      37

**SAVE MODEL AND KERNEL RESULT**

In [ ]:
os.makedirs("genetic_disorder_model", exist_ok=True)
joblib.dump(model, "genetic_disorder_model/final_pipeline.pkl")
joblib.dump(results_df, "genetic_disorder_model/kernel_results.pkl")
print("Saved model and results.")
print("Kernel:", model.named_steps["svm"].kernel)


Saved model and results.
Kernel: linear
Saved model and results.
Kernel: linear


**CREATE STREAMLIT APP**

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.set_page_config(page_title="Genetic Disorder Screening", page_icon="🧬", layout="wide")
st.title("🧬 Genetic Disorder Subclass Screening System")
st.write("SVM-based educational screening and model comparison dashboard.")
st.warning("Educational project only. Predictions are not medical diagnoses.")

model = joblib.load("genetic_disorder_model/final_pipeline.pkl")
results = joblib.load("genetic_disorder_model/kernel_results.pkl")

st.subheader("SVM Kernel Performance")
st.dataframe(results.style.format({"Accuracy":"{:.2%}", "F1-Score":"{:.2%}"}), use_container_width=True)
st.bar_chart(results.set_index("Kernel")[["Accuracy", "F1-Score"]])

st.subheader("Patient Information")
features = list(model.feature_names_in_)
numeric = set(model.named_steps["preprocess"].transformers_[0][2])

with st.form("patient_form"):
    patient = {}
    left, right = st.columns(2)
    for i, name in enumerate(features):
        with (left if i % 2 == 0 else right):
            if name in numeric:
                patient[name] = st.number_input(name, value=0.0, step=1.0, format="%.2f")
            else:
                patient[name] = st.text_input(name, value="")
    submitted = st.form_submit_button("Predict Disorder Subclass")

if submitted:
    row = pd.DataFrame([patient], columns=features)
    for name in numeric:
        row[name] = pd.to_numeric(row[name], errors="coerce")
    for name in set(features) - numeric:
        row[name] = row[name].replace(r"^\s*$", np.nan, regex=True)
    try:
        result = model.predict(row)[0]
        st.subheader("Prediction Result")
        st.success(f"Predicted subclass: {result}")
        st.write("Model: SVM")
        st.write(f"Kernel: {model.named_steps['svm'].kernel}")
        st.info("Educational demonstration only; not a medical diagnosis.")
    except Exception as exc:
        st.error(f"Prediction error: {exc}")


Writing streamlit_app.py


**CHECK REQURIED FILES**

In [ ]:
for p in ["streamlit_app.py", "genetic_disorder_model/final_pipeline.pkl", "genetic_disorder_model/kernel_results.pkl"]:
    print(p, "->", os.path.exists(p))


streamlit_app.py -> True
genetic_disorder_model/final_pipeline.pkl -> True
genetic_disorder_model/kernel_results.pkl -> True


**START STREAMLIT (run only once)**

In [ ]:
import subprocess, time, requests
server = subprocess.Popen([
    "streamlit", "run", "streamlit_app.py",
    "--server.port", "8503", "--server.address", "0.0.0.0",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.headless", "true"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Health:", requests.get("http://127.0.0.1:8503/_stcore/health", timeout=10).text)


Health: ok


**OPEN APP IN COLAB**

In [ ]:
from google.colab import output
url = output.eval_js("google.colab.kernel.proxyPort(8503)")
print(url)


https://8503-m-s-kkb-usc1a2-35l0oi1ckk5b5-a.us-central1-2.prod.colab.dev


YOUR_LINK

In [2]:
!git clone

fatal: You must specify a repository to clone.

usage: git clone [<options>] [--] <repo> [<dir>]

    -v, --[no-]verbose    be more verbose
    -q, --[no-]quiet      be more quiet
    --[no-]progress       force progress reporting
    --[no-]reject-shallow don't clone shallow repository
    -n, --no-checkout     don't create a checkout
    --checkout            opposite of --no-checkout
    --[no-]bare           create a bare repository
    --[no-]mirror         create a mirror repository (implies bare)
    -l, --[no-]local      to clone from a local repository
    --no-hardlinks        don't use local hardlinks, always copy
    --hardlinks           opposite of --no-hardlinks
    -s, --[no-]shared     setup as shared repository
    --[no-]recurse-submodules[=<pathspec>]
                          initialize submodules in the clone
    --[no-]recursive ...  alias of --recurse-submodules
    -j, --[no-]jobs <n>   number of submodules cloned in parallel
    --[no-]template <template-direc

In [4]:
!git clone https://github.com/dhanyarathinam54-ui/Genetic-Disorder-Subclass-Screening-System.git

Cloning into 'Genetic-Disorder-Subclass-Screening-System'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [5]:
!pwd
!ls -lah /content

/content
total 20K
drwxr-xr-x 1 root root 4.0K Nov  3 04:50 .
drwxr-xr-x 1 root root 4.0K Nov  3 04:44 ..
drwxr-xr-x 4 root root 4.0K Oct 26 22:18 .config
drwxr-xr-x 3 root root 4.0K Nov  3 04:50 Genetic-Disorder-Subclass-Screening-System
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


In [6]:
!find /content -maxdepth 4 -type f | head -80

/content/.config/logs/2026.10.26/22.18.52.867204.log
/content/.config/logs/2026.10.26/22.18.38.571729.log
/content/.config/logs/2026.10.26/22.18.15.518759.log
/content/.config/logs/2026.10.26/22.18.44.768076.log
/content/.config/logs/2026.10.26/22.18.53.588181.log
/content/.config/logs/2026.10.26/22.18.45.880512.log
/content/.config/.last_survey_prompt.yaml
/content/.config/active_config
/content/.config/.last_update_check.json
/content/.config/configurations/config_default
/content/.config/default_configs.db
/content/.config/config_sentinel
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/gce
/content/Genetic-Disorder-Subclass-Screening-System/.git/HEAD
/content/Genetic-Disorder-Subclass-Screening-System/.git/logs/HEAD
/content/Genetic-Disorder-Subclass-Screening-System/.git/description
/content/Genetic-Disorder-Subclass-Screening-System/.git/index
/content/Genetic-Disorder-Subclass-Screening-Sys